In [3]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached tiktoken-0.14.0-cp314-cp314-win_amd64.whl.metadata (6.8 kB)
  Using cached langchain_openai-1.7.0-py3-none-any.whl.metadata (3.6 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 24.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 19.2 MB/s  0:00:00
   ---------------------------------------- 0.0/573.0 kB ? eta -:--:--
   ---------------------------------------- 573.0/573.0 kB 19.1 MB/s  0:00:00
   ---------------------------------------- 0.0/862.0 kB ? eta -:--:--
   ---------------------------------------- 862.0/862.0 kB 17.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 24.9 MB/s  0:00:00
   ---------------------------------------- 0.0/9

In [15]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = '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'

In [16]:
os.environ['OPENAI_API_KEY'] = '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'

In [17]:

import os
import bs4
from getpass import getpass

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Set your API key securely
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

#### INDEXING ####

# 1. Load documents from the webpage
loader = WebBaseLoader(
    web_paths=(
        "https://lilianweng.github.io/posts/2023-06-23-agent/",
    ),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()
print(f"Loaded {len(docs)} document(s)")

# 2. Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} chunks")

# 3. Generate embeddings and store them in Chroma
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small")
)

retriever = vectorstore.as_retriever()

#### RETRIEVAL AND GENERATION ####

# 4. Define the RAG prompt locally
prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the following context.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:"""
)

# 5. Initialize the language model
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

# 6. Format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 7. Build the RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 8. Ask a question
answer = rag_chain.invoke("What is Task Decomposition?")
print(answer)


Loaded 1 document(s)
Created 63 chunks


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: lsv2_pt_***************************************64fe. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [11]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."